In [1]:
import pandas as pd
import requests, io

In [2]:
api_url = "https://data.moenv.gov.tw/api/v2/aqx_p_02?api_key=af57253c-e838-46da-a1f5-12b43afd75f3&limit=1000&sort=datacreationdate%20desc&format=CSV"

## 避開SSL認證

In [3]:
resp = requests.get(api_url, verify=False)
print(resp)
df = pd.read_csv(io.StringIO(resp.text))
df

<Response [200]>


c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'data.moenv.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,因受限於資源分配，每日呼叫API的次數不可大於5000次。


In [4]:
df = pd.read_csv(api_url)
df

,因受限於資源分配，每日呼叫API的次數不可大於5000次。


## 清理資料、移除空值、移除重複

In [5]:
## 查看資料結構→ 空值、筆數、型態
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 1 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   因受限於資源分配，每日呼叫API的次數不可大於5000次。  0 non-null      object
dtypes: object(1)
memory usage: 132.0+ bytes


In [6]:
df.describe()

,因受限於資源分配，每日呼叫API的次數不可大於5000次。
count,0
unique,0
top,NaN
freq,NaN


In [7]:
# 查看重複值
# df.duplicated()
# df[df.duplicated(subset=["site", "datacreationdate"])]

# df.drop_duplicates()
df.drop_duplicates(subset=["site", "datacreationdate"])

,因受限於資源分配，每日呼叫API的次數不可大於5000次。


### .dropna() 移除空值

In [8]:
df.drop_duplicates(subset=["site", "datacreationdate"]).dropna()

,因受限於資源分配，每日呼叫API的次數不可大於5000次。


In [9]:
df1 = df.drop_duplicates(subset=["site", "datacreationdate"]).dropna()
df1

,因受限於資源分配，每日呼叫API的次數不可大於5000次。


## sqlite 建立資料庫
- unique(site, datacreationdate)
    - 插入資料唯一的約束
- 整數寫法→ integer
    - 不能寫 int
- autoincrement
    - 不是寫 auto_increment

In [10]:
import sqlite3

In [11]:
sqlstr = '''
create table if not exists data(
id integer primary key autoincrement,
site text,
county text,
pm25 integer,
datacreationdate text,
itemunit text,
unique(site, datacreationdate)
)
'''

In [12]:
conn = sqlite3.connect("pm25.db")
cursor = conn.cursor()
conn, cursor

(<sqlite3.Connection at 0x22936d6c040>, <sqlite3.Cursor at 0x22936d54d40>)

In [13]:
cursor.execute(sqlstr)
conn.commit()

### 插入資料
- or ignore
    - 忽略重複資料


In [14]:
sqlstr = "insert or ignore into data (site,county,pm25,datacreationdate,itemunit)\
    values(?, ?, ?, ?, ?)"

In [15]:
# df1.values
df1.values.tolist()

[]

https://inloop.github.io/sqlite-viewer/

In [16]:
# 插入多筆 .executemany
cursor.executemany(sqlstr, df1.values.tolist())
conn.commit()

In [17]:
# cursor.rowcount 查看更新幾筆資料
cursor.rowcount

0

In [18]:
conn.close()